In [1]:
!pip install -q openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip install -q fastapi uvicorn python-multipart nest-asyncio

In [3]:
import whisper
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = whisper.load_model("large-v3", device=device)

print("Device:", device)

100%|█████████████████████████████████████| 2.88G/2.88G [00:33<00:00, 92.6MiB/s]


Device: cuda


In [4]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import tempfile
import os

app = FastAPI()


@app.get("/")
def root():
    return {
        "status": "running",
        "service": "Arabic Whisper STT"
    }


@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):

    # Create temporary file
    suffix = os.path.splitext(file.filename)[1] or ".wav"

    with tempfile.NamedTemporaryFile(
        delete=False,
        suffix=suffix
    ) as temp:
        temp.write(await file.read())
        audio_path = temp.name

    try:
        # Transcribe
        result = model.transcribe(
            audio_path,
            language="ar"
        )

        text = result["text"].strip()

        return JSONResponse({
            "success": True,
            "text": text
        })

    except Exception as e:

        return JSONResponse(
            {
                "success": False,
                "error": str(e)
            },
            status_code=500
        )

    finally:
        # Delete temporary audio
        if os.path.exists(audio_path):
            os.remove(audio_path)

In [5]:
import threading
import uvicorn

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("🚀 FastAPI server started on port 8000")

🚀 FastAPI server started on port 8000


# **Expose Endpoint**

In [6]:
!pip install -q pyngrok

In [7]:
from pyngrok import ngrok

ngrok.set_auth_token("2d0CfAo24wuKoN7qtN1nxt8NxRx_KA9RFeK5XryniDhTaTVu")

In [8]:
public_url = ngrok.connect(8000)

print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://4563-34-16-225-229.ngrok-free.app" -> "http://localhost:8000"


# **Testing**

In [9]:
import requests

response = requests.get("http://127.0.0.1:8000/")
print(response.json())

INFO:     127.0.0.1:54014 - "GET / HTTP/1.1" 200 OK
{'status': 'running', 'service': 'Arabic Whisper STT'}


In [12]:
import requests

audio_path = "/content/Recording (6).m4a"

url = "https://4563-34-16-225-229.ngrok-free.app/transcribe"

with open(audio_path, "rb") as f:
    response = requests.post(
        url,
        files={
            "file": (
                "audio.wav",
                f,
                "audio/wav"
            )
        }
    )

print(response.status_code)
print(response.json())

INFO:     34.16.225.229:0 - "POST /transcribe HTTP/1.1" 200 OK
200
{'success': True, 'text': 'انا النهاردة ركبت اوبر بمئة جنيه.'}
